# 09 — FLORES-Derived Feature Ablation on MGSM

**Research question:** Do general-purpose language features (identified on out-of-domain FLORES text) interfere with reasoning, or only MGSM-specific features?

**Method:**
1. Run Phase 1 feature-ID methods (monolinguality + probe) on FLORES-200 activations
2. Compute A∩B intersection → FLORES-derived language feature candidates
3. Compare FLORES-derived vs MGSM-derived candidates (Jaccard overlap)
4. Ablate FLORES-derived features on MGSM math problems
5. Compare accuracy to MGSM-derived ablation (Phase 2b H1 results)

**Interpretation:**
- If FLORES-derived features also improve reasoning when ablated → interference is from general language encoding
- If only MGSM-derived features help → interference is domain-specific

## 0. Setup

In [1]:
!pip install -q \
    "transformers>=4.49.0" \
    "sae-lens>=5.9.0" \
    "nnsight>=0.4.0" \
    huggingface-hub \
    scikit-learn \
    numpy \
    pandas \
    matplotlib \
    seaborn \
    tqdm \
    python-dotenv \
    langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 56.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.6/296.6 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.7/963.7 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 24.6 MB/s eta 0:00:00
 

In [2]:
import os
import sys
from pathlib import Path

DRIVE_RESULTS = None

In [3]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'

# Mount Drive first so results are saved even if git clone fails
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
    DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)

    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if IN_COLAB:
    REPO_DIR = '/content/nlp-project'
    if not Path(REPO_DIR).exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
    os.chdir(REPO_DIR)
    # !pip install -q 'numpy>=2.0' datasets -e .
    # Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
else:
    REPO_DIR = os.getcwd()
    from dotenv import load_dotenv
    load_dotenv()
    assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing in .env at repo root'

import sys
sys.path.insert(0, "/content/nlp-project")

# Evict cached src.* so re-imports pick up the latest code
for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

Mounted at /content/drive
Cloning into '/content/nlp-project'...
remote: Enumerating objects: 335, done.
remote: Counting objects: 100% (335/335), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 335 (delta 203), reused 238 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (335/335), 19.49 MiB | 18.18 MiB/s, done.
Resolving deltas: 100% (203/203), done.


In [4]:
import sys
sys.path.insert(0, "/content/nlp-project/")
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm

from src.config import (
    TARGET_LANGUAGES, SAE_WIDTH_16K,
    RESULTS_DIR as _LOCAL_RESULTS_DIR, D_MODEL,
)
from src.model import load_model_and_tokenizer, load_sae
from src.extraction import extract_residual_activations, encode_activations_through_sae
from src.monolinguality import (
    compute_monolinguality, identify_language_features,
    train_language_probe, probe_language_features,
)
from src.intervention import (
    get_sae_decoder_directions, run_generate_with_hooks,
)
from src.data import load_mgsm, parse_answer_number, compute_accuracy

torch.manual_seed(0)
np.random.seed(0)

PRIMARY_LAYER = 17
TOP_K_FEATURE_ID = 50  # top-k per method for feature identification

if DRIVE_RESULTS is not None:
    RESULTS_DIR = DRIVE_RESULTS
    print(f'Saving results to Drive: {RESULTS_DIR}')
else:
    RESULTS_DIR = _LOCAL_RESULTS_DIR
    print(f'Saving results locally: {RESULTS_DIR}')

FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True, parents=True)

Saving results to Drive: /content/drive/MyDrive/nlp-project-results


In [9]:
from pathlib import Path
RESULTS_DIR = Path("/content")
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True, parents=True)

# verify
expected = ["phase1_features.pt", "phase2_ablation.pt", "phase2_zhao_baseline.pt", "phase3_interaction.pt"]
for f in expected:
    p = RESULTS_DIR / f
    print(f"{'✓' if p.exists() else '✗'} {p}")

✓ /content/phase1_features.pt
✓ /content/phase2_ablation.pt
✓ /content/phase2_zhao_baseline.pt
✓ /content/phase3_interaction.pt


## 1. Load model, SAE, and prior results

In [10]:
model, tokenizer = load_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sae, _, _ = load_sae(
    layer=PRIMARY_LAYER,
    width=SAE_WIDTH_16K,
    l0_target='medium',
)
print(f'SAE loaded: layer {PRIMARY_LAYER}, width {SAE_WIDTH_16K}')

# Phase 1 (MGSM-derived features)
phase1 = torch.load(RESULTS_DIR / 'phase1_features.pt', weights_only=False)
mgsm_intersection = phase1['intersection_features'][PRIMARY_LAYER]
print('\nPhase 1 MGSM A\u2229B features at L17:')
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: {len(mgsm_intersection[lang])} features')

# Phase 2b (baseline + H1 results for comparison)
phase2a = torch.load(RESULTS_DIR / 'phase2_zhao_baseline.pt', weights_only=False)
phase2b = torch.load(RESULTS_DIR / 'phase2_ablation.pt', weights_only=False)
baseline_results = phase2a['baseline_results']
h1_test_results = phase2b['h1_test']
best_k = phase2b['best_k']
print(f'\nPhase 2b best k: {best_k}')
print('MGSM-derived H1 test results (k={best_k}):')
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: {h1_test_results[lang]["accuracy"]:.3f}')

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

SAE loaded: layer 17, width 16384

Phase 1 MGSM A∩B features at L17:
  en: 19 features
  zh: 6 features
  es: 9 features
  bn: 16 features
  sw: 11 features

Phase 2b best k: 20
MGSM-derived H1 test results (k={best_k}):
  en: 0.712
  zh: 0.596
  es: 0.636
  bn: 0.480
  sw: 0.296


## 2. Load FLORES-200 and extract SAE features

In [11]:
from datasets import load_dataset

FLORES_LANG_MAP = {
    'en': 'eng_Latn',
    'zh': 'cmn_Hans',
    'es': 'spa_Latn',
    'bn': 'ben_Beng',
    'sw': 'swh_Latn',
}

flores_texts = {}
for lang, flores_code in FLORES_LANG_MAP.items():
    ds = load_dataset('openlanguagedata/flores_plus', flores_code, split='devtest')
    flores_texts[lang] = ds['text']
    print(f'  {lang} ({flores_code}): {len(flores_texts[lang])} sentences')

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

...

swh_Latn.jsonl: 0.00B [00:00, ?B/s]

swh_Latn.jsonl: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

  sw (swh_Latn): 1012 sentences


In [12]:
def make_chat_prompt(text):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': text}],
        tokenize=False,
        add_generation_prompt=True,
    )

flores_prompts = {
    lang: [make_chat_prompt(s) for s in sents]
    for lang, sents in flores_texts.items()
}
print(f'Total FLORES prompts: {sum(len(v) for v in flores_prompts.values())}')

Total FLORES prompts: 5060


In [13]:
# Extract residual activations at L17 for all FLORES sentences
flores_residuals = {}
for lang in TARGET_LANGUAGES:
    print(f'\nExtracting {lang} ({len(flores_prompts[lang])} sentences)...')
    acts = extract_residual_activations(
        model, tokenizer,
        flores_prompts[lang],
        layers=[PRIMARY_LAYER],
        positions='last',
    )
    flores_residuals[lang] = acts[PRIMARY_LAYER]
    print(f'  {lang}: {flores_residuals[lang].shape}')

# Encode through SAE
flores_feats = {}
for lang in TARGET_LANGUAGES:
    flores_feats[lang] = encode_activations_through_sae(
        flores_residuals[lang], sae,
    )
    print(f'  {lang} SAE features: {flores_feats[lang].shape}')

del flores_residuals
torch.cuda.empty_cache() if torch.cuda.is_available() else None


Extracting en (1012 sentences)...


Extracting activations: 100%|██████████| 253/253 [00:21<00:00, 11.57it/s]


  en: torch.Size([1012, 2560])

Extracting zh (1012 sentences)...


Extracting activations: 100%|██████████| 253/253 [00:20<00:00, 12.13it/s]


  zh: torch.Size([1012, 2560])

Extracting es (1012 sentences)...


...

  es: torch.Size([1012, 2560])

Extracting bn (1012 sentences)...


Extracting activations: 100%|██████████| 253/253 [00:20<00:00, 12.23it/s]


  bn: torch.Size([1012, 2560])

Extracting sw (1012 sentences)...


Extracting activations: 100%|██████████| 253/253 [00:20<00:00, 12.51it/s]


  sw: torch.Size([1012, 2560])
  en SAE features: torch.Size([1012, 16384])
  zh SAE features: torch.Size([1012, 16384])
  es SAE features: torch.Size([1012, 16384])
  bn SAE features: torch.Size([1012, 16384])
  sw SAE features: torch.Size([1012, 16384])


## 3. Phase 1 methods on FLORES features

Run the same two feature-identification methods used on MGSM, but on FLORES activations:
- **Method A**: Monolinguality metric ν → top-50 per language
- **Method B**: Language probe → top-50 per language
- **A∩B**: Intersection of both methods

In [14]:
# Method A: monolinguality on FLORES
mono_flores = compute_monolinguality(flores_feats)
method_a_flores = identify_language_features(mono_flores, top_k=TOP_K_FEATURE_ID)

print('Method A (monolinguality) top-1 \u03bd per language on FLORES:')
for lang in TARGET_LANGUAGES:
    top_feat = method_a_flores[lang][0]
    top_nu = mono_flores[lang][top_feat].item()
    print(f'  {lang}: f{top_feat} (\u03bd = {top_nu:.1f})')

Method A (monolinguality) top-1 ν per language on FLORES:
  en: f34 (ν = 209.6)
  zh: f828 (ν = 333.7)
  es: f497 (ν = 112.6)
  bn: f1066 (ν = 394.0)
  sw: f356 (ν = 837.0)


In [15]:
# Method B: language probe on FLORES
print('Training language probe on FLORES SAE features...')
flores_probe, flores_importances = train_language_probe(flores_feats)
languages_sorted = sorted(TARGET_LANGUAGES)

# In-sample accuracy
X_flores = np.concatenate([flores_feats[l].float().numpy() for l in languages_sorted])
y_flores = np.concatenate([np.full(flores_feats[l].shape[0], i)
                            for i, l in enumerate(languages_sorted)])
flores_probe_acc = float(np.mean(flores_probe.predict(X_flores) == y_flores))
print(f'  FLORES in-sample probe accuracy: {flores_probe_acc:.4f}')

method_b_flores = probe_language_features(
    flores_probe, flores_importances, languages_sorted, top_k=TOP_K_FEATURE_ID
)

Training language probe on FLORES SAE features...
  FLORES in-sample probe accuracy: 0.9231


In [16]:
# A \u2229 B intersection on FLORES
flores_intersection = {}
print('FLORES A\u2229B intersection sizes:')
for lang in TARGET_LANGUAGES:
    set_a = set(method_a_flores[lang])
    set_b = set(method_b_flores[lang])
    isect = sorted(set_a & set_b)
    flores_intersection[lang] = isect
    jaccard = len(isect) / len(set_a | set_b) if (set_a | set_b) else 0
    print(f'  {lang}: {len(isect)} features (Jaccard = {jaccard:.3f})')

FLORES A∩B intersection sizes:
  en: 12 features (Jaccard = 0.136)
  zh: 9 features (Jaccard = 0.099)
  es: 10 features (Jaccard = 0.111)
  bn: 11 features (Jaccard = 0.124)
  sw: 13 features (Jaccard = 0.149)


## 4. Compare FLORES-derived vs MGSM-derived features

In [17]:
print('Cross-domain feature overlap (FLORES A\u2229B vs MGSM A\u2229B):')
print(f'{"lang":>4}  {"MGSM":>5}  {"FLORES":>6}  {"overlap":>7}  {"Jaccard":>8}')
print('-' * 40)

overlap_results = {}
for lang in TARGET_LANGUAGES:
    mgsm_set = set(mgsm_intersection[lang])
    flores_set = set(flores_intersection[lang])
    overlap = mgsm_set & flores_set
    union = mgsm_set | flores_set
    jaccard = len(overlap) / len(union) if union else 0
    overlap_results[lang] = {
        'mgsm_count': len(mgsm_set),
        'flores_count': len(flores_set),
        'overlap_count': len(overlap),
        'overlap_features': sorted(overlap),
        'jaccard': jaccard,
    }
    print(f'{lang:>4}  {len(mgsm_set):>5}  {len(flores_set):>6}  '
          f'{len(overlap):>7}  {jaccard:>8.3f}')

Cross-domain feature overlap (FLORES A∩B vs MGSM A∩B):
lang   MGSM  FLORES  overlap   Jaccard
----------------------------------------
  en     19      12        4     0.148
  zh      6       9        5     0.500
  es      9      10        3     0.188
  bn     16      11        7     0.350
  sw     11      13        7     0.412


## 5. Ablate FLORES-derived features on MGSM

Using directional ablation at L17 (same method as Phase 2b H1), but with FLORES-derived A∩B features instead of MGSM-derived ones.

In [18]:
# Load MGSM data
mgsm = load_mgsm(TARGET_LANGUAGES)
mgsm_prompts = {
    lang: [make_chat_prompt(ex['question']) for ex in mgsm[lang]]
    for lang in TARGET_LANGUAGES
}
mgsm_gold = {
    lang: [ex['answer_number'] for ex in mgsm[lang]]
    for lang in TARGET_LANGUAGES
}
print(f'MGSM loaded: {sum(len(v) for v in mgsm_prompts.values())} prompts')

MGSM loaded: 1250 prompts


In [ ]:
# Ablation with FLORES-derived features
# Use the same k as Phase 2b for fair comparison
K_ABLATE = best_k  # from Phase 2b

flores_ablation_results = {}

for lang in TARGET_LANGUAGES:
    # Get FLORES-derived features (up to k)
    flores_feats_lang = flores_intersection[lang][:K_ABLATE]

    if len(flores_feats_lang) == 0:
        print(f'{lang}: no FLORES A\u2229B features, skipping')
        flores_ablation_results[lang] = {
            'accuracy': baseline_results[lang]['accuracy'],
            'n_features': 0,
            'features': [],
        }
        continue

    # If fewer than k features available, use all of them
    actual_k = len(flores_feats_lang)
    print(f'\n{lang}: ablating {actual_k} FLORES-derived features at L17')
    print(f'  Features: {flores_feats_lang}')

    # Get decoder directions
    directions = get_sae_decoder_directions(sae, flores_feats_lang)
    ablation_config = {PRIMARY_LAYER: directions}

    # Generate with ablation hooks
    outputs = run_generate_with_hooks(
        model, tokenizer,
        mgsm_prompts[lang],
        ablation_config=ablation_config,
        positions='last',
        max_new_tokens=512,
    )

    # Parse and evaluate
    predictions = [parse_answer_number(o) for o in outputs]
    accuracy = compute_accuracy(predictions, mgsm_gold[lang])
    correct = [
        p is not None and abs(p - g) < 1e-6
        for p, g in zip(predictions, mgsm_gold[lang])
    ]

    flores_ablation_results[lang] = {
        'accuracy': accuracy,
        'n_features': actual_k,
        'features': flores_feats_lang,
        'predictions': predictions,
        'correct': correct,
    }

    baseline_acc = baseline_results[lang]['accuracy']
    delta = accuracy - baseline_acc
    print(f'  Accuracy: {accuracy:.3f} (baseline: {baseline_acc:.3f}, \u0394: {delta:+.3f})')


en: ablating 12 FLORES-derived features at L17
  Features: [117, 166, 191, 486, 864, 3419, 4216, 6854, 7157, 8820, 12003, 12278]


## 6. Comparison: FLORES-derived vs MGSM-derived ablation

In [ ]:
print('\n' + '=' * 75)
print('COMPARISON: FLORES-derived vs MGSM-derived feature ablation on MGSM')
print('=' * 75)
print(f'\n{"Lang":>4}  {"Baseline":>8}  {"MGSM-abl":>9} ({"\u0394":>6})  '
      f'{"FLORES-abl":>10} ({"\u0394":>6})  {"Overlap":>7}')
print('-' * 70)

comparison_rows = []
for lang in TARGET_LANGUAGES:
    base = baseline_results[lang]['accuracy']
    mgsm_acc = h1_test_results[lang]['accuracy']
    flores_acc = flores_ablation_results[lang]['accuracy']
    mgsm_delta = mgsm_acc - base
    flores_delta = flores_acc - base
    olap = overlap_results[lang]['overlap_count']

    comparison_rows.append({
        'lang': lang,
        'baseline': base,
        'mgsm_ablation': mgsm_acc,
        'mgsm_delta': mgsm_delta,
        'flores_ablation': flores_acc,
        'flores_delta': flores_delta,
        'feature_overlap': olap,
        'flores_n_features': flores_ablation_results[lang]['n_features'],
    })

    print(f'{lang:>4}  {base:>8.3f}  {mgsm_acc:>9.3f} ({mgsm_delta:>+6.3f})  '
          f'{flores_acc:>10.3f} ({flores_delta:>+6.3f})  {olap:>7}')

# Averages
avg_base = np.mean([r['baseline'] for r in comparison_rows])
avg_mgsm = np.mean([r['mgsm_ablation'] for r in comparison_rows])
avg_flores = np.mean([r['flores_ablation'] for r in comparison_rows])
print('-' * 70)
print(f'{"avg":>4}  {avg_base:>8.3f}  {avg_mgsm:>9.3f} ({avg_mgsm-avg_base:>+6.3f})  '
      f'{avg_flores:>10.3f} ({avg_flores-avg_base:>+6.3f})')

In [ ]:
# Interpretation
print('\n=== Interpretation ===')
print()
mgsm_gains = [r['mgsm_delta'] for r in comparison_rows]
flores_gains = [r['flores_delta'] for r in comparison_rows]

if np.mean(flores_gains) > 0.01:
    print('FLORES-derived features ALSO improve reasoning when ablated.')
    print('-> Interference is from GENERAL language encoding, not domain-specific.')
elif np.mean(flores_gains) < -0.01:
    print('FLORES-derived features HURT reasoning when ablated.')
    print('-> General language features are entangled with reasoning (SHARED-like).')
    print('-> Only MGSM-specific features cleanly separate language from reasoning.')
else:
    print('FLORES-derived features have NEGLIGIBLE effect on reasoning.')
    print('-> The interfering features are domain-specific to math reasoning.')
    print('-> General language features neither help nor hurt.')

# Per-language breakdown
print()
for lang, mg, fg in zip(TARGET_LANGUAGES, mgsm_gains, flores_gains):
    if fg > 0.02:
        verdict = 'FLORES helps'
    elif fg < -0.02:
        verdict = 'FLORES hurts'
    else:
        verdict = 'FLORES neutral'
    print(f'  {lang}: MGSM {mg:+.3f}, FLORES {fg:+.3f} -> {verdict}')

## 7. Figures

In [ ]:
sns.set_theme(style='whitegrid', context='paper', font_scale=1.1)

# Figure 1: Grouped bar chart — baseline / MGSM-derived / FLORES-derived
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(TARGET_LANGUAGES))
width = 0.25

base_vals = [baseline_results[l]['accuracy'] for l in TARGET_LANGUAGES]
mgsm_vals = [h1_test_results[l]['accuracy'] for l in TARGET_LANGUAGES]
flores_vals = [flores_ablation_results[l]['accuracy'] for l in TARGET_LANGUAGES]

bars1 = ax.bar(x - width, base_vals, width, label='Unmodified', color='#a0a0a0')
bars2 = ax.bar(x, mgsm_vals, width, label=f'MGSM-derived (k={best_k})', color='#c0392b')
bars3 = ax.bar(x + width, flores_vals, width, label='FLORES-derived', color='#2980b9')

ax.set_ylabel('MGSM Accuracy')
ax.set_xlabel('Language')
ax.set_xticks(x)
ax.set_xticklabels(TARGET_LANGUAGES)
ax.set_ylim(0, 1.0)
ax.legend(loc='upper right')
ax.set_title('MGSM-derived vs FLORES-derived feature ablation')

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=7, padding=2)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_flores_vs_mgsm_ablation.png', dpi=150)
plt.show()

In [ ]:
# Figure 2: Feature overlap Venn-style bar chart
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(TARGET_LANGUAGES))
width = 0.3

mgsm_only = [overlap_results[l]['mgsm_count'] - overlap_results[l]['overlap_count']
             for l in TARGET_LANGUAGES]
both = [overlap_results[l]['overlap_count'] for l in TARGET_LANGUAGES]
flores_only = [overlap_results[l]['flores_count'] - overlap_results[l]['overlap_count']
               for l in TARGET_LANGUAGES]

ax.bar(x, mgsm_only, width=0.6, label='MGSM-only', color='#c0392b', alpha=0.7)
ax.bar(x, both, width=0.6, bottom=mgsm_only, label='Shared', color='#8e44ad', alpha=0.7)
ax.bar(x, flores_only, width=0.6,
       bottom=[m + b for m, b in zip(mgsm_only, both)],
       label='FLORES-only', color='#2980b9', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(TARGET_LANGUAGES)
ax.set_ylabel('Number of A\u2229B features')
ax.set_title('Feature set overlap: MGSM-derived vs FLORES-derived')
ax.legend()

# Annotate Jaccard
for i, lang in enumerate(TARGET_LANGUAGES):
    total = mgsm_only[i] + both[i] + flores_only[i]
    j = overlap_results[lang]['jaccard']
    ax.text(i, total + 0.5, f'J={j:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_feature_overlap.png', dpi=150)
plt.show()

In [ ]:
# Figure 3: Delta comparison (MGSM-derived vs FLORES-derived)
fig, ax = plt.subplots(figsize=(6, 4))

for i, lang in enumerate(TARGET_LANGUAGES):
    ax.scatter(comparison_rows[i]['mgsm_delta'],
              comparison_rows[i]['flores_delta'],
              s=120, zorder=5, label=lang)

# Reference lines
lims = [-0.15, 0.20]
ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
ax.axhline(0, color='grey', alpha=0.3, lw=0.5)
ax.axvline(0, color='grey', alpha=0.3, lw=0.5)

ax.set_xlabel('\u0394 accuracy (MGSM-derived ablation)')
ax.set_ylabel('\u0394 accuracy (FLORES-derived ablation)')
ax.set_title('Per-language accuracy delta: MGSM vs FLORES features')
ax.legend()
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delta_comparison.png', dpi=150)
plt.show()

## 8. Save results

In [ ]:
payload = {
    'config': {
        'primary_layer': PRIMARY_LAYER,
        'sae_width': SAE_WIDTH_16K,
        'top_k_feature_id': TOP_K_FEATURE_ID,
        'k_ablate': K_ABLATE,
        'flores_lang_map': FLORES_LANG_MAP,
    },
    'flores_method_a': {lang: method_a_flores[lang] for lang in TARGET_LANGUAGES},
    'flores_method_b': {lang: method_b_flores[lang] for lang in TARGET_LANGUAGES},
    'flores_intersection': flores_intersection,
    'flores_probe_accuracy': flores_probe_acc,
    'overlap_results': overlap_results,
    'flores_ablation_results': {
        lang: {
            'accuracy': r['accuracy'],
            'n_features': r['n_features'],
            'features': r['features'],
        }
        for lang, r in flores_ablation_results.items()
    },
    'comparison': comparison_rows,
}

out = RESULTS_DIR / 'phase_flores_ablation.pt'
torch.save(payload, out)
print(f'Saved to {out}')
print(f'Figures saved to {FIG_DIR}/')